In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='120cfa19-7e81-4530-8c0c-1712ce1c6197'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'x4jampz64', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 243, 'total_tokens': 262, 'completion_time': 0.046010451, 'completion_tokens_details': None, 'prompt_time': 0.125391309, 'prompt_tokens_details': None, 'queue_time': 0.106576378, 'total_time': 0.17140176}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4cf8-c57b-7df3-81a3-e3ac0a1350a7-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='133dac34-da0e-4dd6-bd2d-8b765db04750'),
              AIMessage(content="I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 244, 'total_tokens': 269, 'completion_time': 0.073684674, 'completion_tokens_details': None, 'prompt_time': 0.026501398, 'prompt_tokens_details': None, 'queue_time': 0.13181987, 'total_time': 0.100186072}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4cf8-c900-7b22-811a-7310c35e7d1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 244, 'output_tokens': 25, 'total_tokens': 269})]}


## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is turquoise blue")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'turquoise blue',
 'messages': [HumanMessage(content='My favourite colour is turquoise blue', additional_kwargs={}, response_metadata={}, id='2ad018d8-8c94-444d-b98b-565203769553'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0e35z6871', 'function': {'arguments': '{"favourite_colour":"turquoise blue"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 298, 'total_tokens': 319, 'completion_time': 0.069760853, 'completion_tokens_details': None, 'prompt_time': 0.016201127, 'prompt_tokens_details': None, 'queue_time': 0.270358739, 'total_time': 0.08596198}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4cf8-cb12-7801-9249-cdcaf8e9b676-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'fa

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'turquoise blue',
 'messages': [HumanMessage(content='My favourite colour is turquoise blue', additional_kwargs={}, response_metadata={}, id='2ad018d8-8c94-444d-b98b-565203769553'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0e35z6871', 'function': {'arguments': '{"favourite_colour":"turquoise blue"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 298, 'total_tokens': 319, 'completion_time': 0.069760853, 'completion_tokens_details': None, 'prompt_time': 0.016201127, 'prompt_tokens_details': None, 'queue_time': 0.270358739, 'total_time': 0.08596198}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4cf8-cb12-7801-9249-cdcaf8e9b676-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'fa